# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "milagros_1", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_milagros_1/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció? --> app_version
2. ¿Qué valor de `event_type` no existía? --> refund
3. ¿Cuántas filas e identificadores distintos contiene? --> 10000

In [0]:
#TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = spark.read.json(challenge_path)

challenge.printSchema()

print("Cantidad de filas:", challenge.count())
print("Event IDs distintos:", challenge.select("event_id").distinct().count())

challenge.select("event_type").distinct().show()

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)

Cantidad de filas: 10001
Event IDs distintos: 10000
+----------+
|event_type|
+----------+
|  checkout|
|    refund|
|      view|
+----------+



## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

historical = spark.table(source_table)

historical = historical.withColumn(
    "app_version",
    F.lit(None).cast("string")
)

new_events = (
    challenge
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(challenge_path))
    .withColumn("_ingested_at", F.current_timestamp())
)

combined = historical.unionByName(
    new_events,
    allowMissingColumns=True
)

deduplicated = combined.dropDuplicates(["event_id"])

(
    deduplicated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Total final:", spark.table(target_table).count())
print(
    "Event IDs distintos:",
    spark.table(target_table).select("event_id").distinct().count()
)

Total final: 210000
Event IDs distintos: 210000


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:

result = spark.table(target_table)
total_rows = None
distinct_ids = None
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.


Detectar una evolución de esquema significa identificar que la estructura de los datos cambió, por ejemplo, porque apareció una nueva columna como app_version. Aceptarla técnicamente implica adaptar el proceso para poder incorporar ese cambio sin perder información previa ni romper la ingesta, manteniendo compatibilidad con los datos históricos. En este caso, se agregó la nueva columna y se utilizó NULL para los registros anteriores. Sin embargo, que un cambio pueda incorporarse técnicamente no significa que sea correcto para el negocio. Validarlo desde el negocio requiere entender qué representa el nuevo campo o valor, como refund, y confirmar que tenga sentido dentro del proceso y pueda utilizarse correctamente. Por eso, la evolución del esquema primero se detecta y se controla técnicamente, pero también necesita una validación de su significado.